<!-- source: new + K:Warsztaty_Krzysztof/rag_agent/notebooks/01_parse_robotics_documents.py -->
# Wzorzec M3 · RAG od PDF do wyszukiwania, na raportach o robotyce

**Forma:** demo prowadzącego (Krzysztof, workspace Premium)

**Po co to demo:** uczestnicy za chwilę zbudują RAG na raportach TechRetail. Tu widzą cały mechanizm na innym korpusie: 10 fikcyjnych, trzystronicowych artykułów o robotyce po angielsku, z tabelami, wykresami i ilustracjami. Najważniejsza lekcja wynika z porównania: **rozmiar fragmentu zależy od dokumentów**, a nie od tutoriala.

| Krok | Robotyka (to demo) | TechRetail (lab) |
|---|---|---|
| Dokumenty | 3 strony, dużo ilustracji | 5 stron, tabele i wykresy |
| Chunking | 2000 / 200 | 600 / 100 |
| Indeks | `robotics_chunks_index` | `retail_rag_chunks_index` |
| Managed RAG | Knowledge Assistant z Guidelines | Knowledge Assistant (demo) |

Na podstawie modułu `rag_agent` (notebooki 01–05) Krzysztofa, zweryfikowanego na Free Edition 22.07.2026 (bez rerankera i Knowledge Assistant). Pliki PDF leżą w repozytorium: `Warsztaty_Krzysztof/rag_agent/documents`.

**Przygotuj dzień wcześniej:** uruchom cały notebook raz, bo parsowanie i pierwszy indeks trwają długo. Na warsztacie przejdź przez komórki, a długie kroki pokaż z gotowymi wynikami.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new + K:Warsztaty_Krzysztof/rag_agent/notebooks/01_parse_robotics_documents.py
import json
import os
import re
import shutil
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

ROBOTICS_VOLUME = "robotics_files"
ROBOTICS_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{ROBOTICS_VOLUME}"
ROBOTICS_PAGES = f"{ROBOTICS_PATH}/parsed_pages"
ROBOTICS_PARSED = f"{CATALOG}.{SCHEMA}.robotics_parsed_documents"
ROBOTICS_CHUNKS = f"{CATALOG}.{SCHEMA}.robotics_chunks"
ROBOTICS_INDEX = f"{CATALOG}.{SCHEMA}.robotics_chunks_index"
SOURCE_PDFS = Path(os.getcwd()).parent.parent / "Warsztaty_Krzysztof" / "rag_agent" / "documents"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{ROBOTICS_VOLUME}")
for pdf in sorted(SOURCE_PDFS.glob("*.pdf")):
    target = Path(ROBOTICS_PATH) / pdf.name
    if not target.exists():
        shutil.copy(pdf, target)
print(f"{ROBOTICS_PATH}: {len(list(Path(ROBOTICS_PATH).glob('*.pdf')))} PDF")

<!-- source: K:Warsztaty_Krzysztof/rag_agent/notebooks/01_parse_robotics_documents.py -->
## 1. Parsowanie z obrazami stron i opisami wykresów

`imageOutputPath` zapisuje wyrenderowane strony, a `descriptionElementTypes` każe modelowi opisać wykresy i ilustracje. Dzięki temu obraz też staje się tekstem, który da się wyszukać.

In [ ]:
# source: K:Warsztaty_Krzysztof/rag_agent/notebooks/01_parse_robotics_documents.py
if spark.catalog.tableExists(ROBOTICS_PARSED):
    print(f"Używam wcześniej sparsowanych dokumentów: {ROBOTICS_PARSED}")
else:
    spark.sql(f"""
        CREATE TABLE {ROBOTICS_PARSED} AS
        SELECT path, to_json(ai_parse_document(
            content,
            map('version', '2.0', 'imageOutputPath', '{ROBOTICS_PAGES}', 'descriptionElementTypes', '*')
        )) AS parsed_json
        FROM read_files('{ROBOTICS_PATH}', format => 'binaryFile', fileNamePattern => '*.pdf')
    """)
display(spark.sql(f"""
    SELECT regexp_extract(path, '([^/]+)$', 1) AS plik,
           array_size(try_cast(parsed:document:pages AS ARRAY<VARIANT>)) AS strony,
           array_size(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS elementy
    FROM (SELECT path, parse_json(parsed_json) AS parsed FROM {ROBOTICS_PARSED})
    ORDER BY plik
"""))

In [ ]:
# source: WS3[12] + K:Warsztaty_Krzysztof/rag_agent/notebooks/includes/document_renderer.py
import base64
import html
from collections import Counter

TYPE_COLORS = {"title": "#7c3aed", "section_header": "#2563eb", "text": "#16a34a", "table": "#ea580c",
               "figure": "#db2777", "caption": "#0891b2", "page_header": "#6b7280", "page_footer": "#6b7280"}


def render_parsed_page(parsed_json: str, page_no: int = 1, max_width: int = 720) -> None:
    doc = json.loads(parsed_json).get("document", {})
    page = next((p for p in doc.get("pages", []) if int(p.get("id", -1)) == page_no - 1), None)
    image_uri = (page or {}).get("image_uri") or ""
    if not os.path.isfile(image_uri):
        displayHTML(f"<p>Brak obrazu strony {page_no}: <code>{html.escape(image_uri)}</code></p>")
        return
    boxes = [(el, bb["coord"][:4]) for el in doc.get("elements", []) for bb in (el.get("bbox") or [])[:1]
             if bb.get("page_id") == page_no - 1 and len(bb.get("coord") or []) >= 4]
    coords = [c for _, box in boxes for c in box]
    width = max([c for i, c in enumerate(coords) if i % 4 in (0, 2)] or [1000])
    height = max([c for i, c in enumerate(coords) if i % 4 in (1, 3)] or [1400])
    shapes = "".join(
        f'<g><title>{html.escape(str(el.get("type")) + ": " + str(el.get("content") or el.get("description") or "")[:300])}</title>'
        f'<rect x="{min(l, r)}" y="{min(t, b)}" width="{abs(r - l)}" height="{abs(b - t)}" fill="{TYPE_COLORS.get(el.get("type"), "#64748b")}" '
        f'fill-opacity="0.12" stroke="{TYPE_COLORS.get(el.get("type"), "#64748b")}" stroke-width="{max(width, height) * 0.0025}"/></g>'
        for el, (l, t, r, b) in boxes
    )
    legend = ", ".join(f"{kind}: {count}" for kind, count in sorted(Counter(str(el.get("type")) for el, _ in boxes).items()))
    with open(image_uri, "rb") as image:
        image_b64 = base64.b64encode(image.read()).decode("ascii")
    displayHTML(f"""<div style="max-width:{max_width}px;font-family:Arial">
      <p><b>Strona {page_no}</b> · {legend} · najedź na ramkę, żeby zobaczyć treść</p>
      <div style="position:relative"><img src="data:image/png;base64,{image_b64}" style="width:100%;display:block">
      <svg viewBox="0 0 {width} {height}" preserveAspectRatio="none" style="position:absolute;inset:0;width:100%;height:100%">{shapes}</svg></div></div>""")


sample = spark.table(ROBOTICS_PARSED).orderBy("path").first()
print(sample["path"])
render_parsed_page(sample["parsed_json"], page_no=1)

<!-- source: K:Warsztaty_Krzysztof/rag_agent/notebooks/02_chunking.py + slide 37 -->
## 2. Chunking: ten sam splitter, inne dokumenty, inne parametry

Komórka tnie te same teksty na dwa sposoby i zestawia liczby. Przy trzech stronach 600 znaków daje drobne fragmenty bez kontekstu, a 2000 znaków trzyma sekcję w całości. Przy pięciostronicowych raportach TechRetail jest odwrotnie. **Wniosek dla Twoich danych: zacznij od długości typowej sekcji dokumentu, a nie od liczby z tutoriala.**

In [ ]:
# source: K:Warsztaty_Krzysztof/rag_agent/notebooks/02_chunking.py
from langchain_text_splitters import RecursiveCharacterTextSplitter


def to_plain_text(parsed_json: str) -> str:
    doc = json.loads(parsed_json).get("document") or {}
    by_page = {}
    for element in sorted(doc.get("elements") or [], key=lambda e: e.get("id", 0)):
        text = re.sub(r"<[^>]+>", " ", str(element.get("content") or element.get("description") or "")).strip()
        if text:
            bbox = element.get("bbox") or []
            by_page.setdefault(int(bbox[0].get("page_id", 0)) if bbox else 0, []).append(text)
    return "\n== page ==\n".join("\n".join(by_page[page]) for page in sorted(by_page))


documents = spark.table(ROBOTICS_PARSED).toPandas()
documents["text"] = documents["parsed_json"].map(to_plain_text)
separators = ["\n== page ==\n", "\n\n", "\n", " ", ""]
comparison = []
for size, overlap in ((2000, 200), (600, 100)):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap, separators=separators)
    pieces = [len(chunk) for text in documents["text"] for chunk in splitter.split_text(text)]
    comparison.append({"parametry": f"{size}/{overlap}", "fragmentów": len(pieces), "średnio znaków": round(sum(pieces) / len(pieces))})
display(pd.DataFrame(comparison))

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200, separators=separators)
chunks = pd.DataFrame([
    {"doc_id": Path(row.path).stem, "chunk_position": i, "content": chunk}
    for row in documents.itertuples() for i, chunk in enumerate(splitter.split_text(row.text))
])
chunks["chunk_id"] = chunks["doc_id"] + "_" + chunks["chunk_position"].astype(str)
spark.createDataFrame(chunks).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(ROBOTICS_CHUNKS)
spark.sql(f"ALTER TABLE {ROBOTICS_CHUNKS} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"{ROBOTICS_CHUNKS}: {len(chunks)} fragmentów (2000/200)")

<!-- source: K:Warsztaty_Krzysztof/rag_agent/notebooks/03_vector_search.py -->
## 3. Indeks i trzy tryby wyszukiwania

Ten sam endpoint AI Search co w module (`retail_rag_search`), drugi indeks. Na Free Edition pierwszy indeks przez kilka minut pokazuje `PROVISIONING_ENDPOINT`, a `sync()` zaraz po utworzeniu zwraca „index is not ready”. Wystarczy ponowić.

In [ ]:
# source: K:Warsztaty_Krzysztof/rag_agent/notebooks/03_vector_search.py
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)
if not search_client.index_exists(SEARCH_ENDPOINT, ROBOTICS_INDEX):
    search_client.create_delta_sync_index_and_wait(
        endpoint_name=SEARCH_ENDPOINT, index_name=ROBOTICS_INDEX, primary_key="chunk_id",
        source_table_name=ROBOTICS_CHUNKS, pipeline_type="TRIGGERED",
        embedding_source_column="content", embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
        columns_to_sync=["doc_id", "chunk_position"], verbose=True,
    )
index = search_client.get_index(SEARCH_ENDPOINT, ROBOTICS_INDEX)

rows = []
for question, mode in (("How do mobile robots avoid obstacles?", "ANN"),
                       ("How do collaborative robots work safely with people?", "HYBRID"),
                       ("sensors and actuators", "FULL_TEXT")):
    try:
        result = index.similarity_search(query_text=question, columns=["doc_id", "chunk_position", "content"],
                                         num_results=3, query_type=mode)
    except Exception as e:  # FULL_TEXT bywa podglądem wyłączonym w workspace (Settings → Previews)
        rows.append({"tryb": mode, "pytanie": question, "fragment": None, "score": None, "początek": f"niedostępny: {str(e)[:120]}"})
        continue
    columns = [c["name"] for c in result["manifest"]["columns"]]
    for record in (dict(zip(columns, r)) for r in result["result"].get("data_array", [])):
        rows.append({"tryb": mode, "pytanie": question, "fragment": f"{record['doc_id']} #{record['chunk_position']}",
                     "score": round(float(record.get("score", 0)), 3), "początek": record["content"][:80]})
display(pd.DataFrame(rows))

<!-- source: K:Warsztaty_Krzysztof/rag_agent/notebooks/05_building_assistant.py -->
## 4. Ten sam RAG bez kodu: Knowledge Assistant z Guidelines

1. **Agents → Create agent → Knowledge Assistant.** Nazwa: `robotics-course-specialist`.
2. **Add knowledge source → Files in a Volume** → `workspace.default.robotics_files`. Opis: „10 fikcyjnych artykułów edukacyjnych o robotyce: czujniki, ruch, roboty mobilne i współpracujące, logistyka, AI, bezpieczeństwo.”
3. **Instructions:** „Odpowiadaj wyłącznie na podstawie materiałów o robotyce. Jeśli ich brakuje, powiedz, że nie wiesz.”
4. **Test your agent:** *„How do mobile robots avoid obstacles?”* → **View sources**.
5. **Examples → + Add:** *„Jak skonfigurować robota Kratos?”* → **Guidelines:** „Poleć aplikację Kratos Control: zaloguj się, Add Robot, parowanie w aplikacji” oraz „Zaznacz, że to wytyczna demonstracyjna, a nie treść materiałów”. Zadaj pytanie ponownie.

**Lekcja:** ekspert poprawia odpowiedzi przez Guidelines bez zmiany dokumentów i bez kodu. Na Free Edition Knowledge Assistant nie jest dostępny.

<!-- source: new + slide 38 -->
## Karta wzorca: dokumenty jako narzędzie agenta

1. **Dokumenty do Volume**, a potem `ai_parse_document`. Wykresy i ilustracje dostają opis tekstowy.
2. **Tekst per strona, fragmenty o długości typowej sekcji** (sprawdź 2–3 warianty na swoich dokumentach).
3. **Tabela fragmentów** z kluczem i metadanymi (`doc_id`, pozycja), z włączonym Change Data Feed.
4. **Indeks Delta Sync** z zarządzanymi embeddingami; zacznij od HYBRID, a filtry dodaj po metadanych.
5. **Odpowiedź tylko z fragmentów, z cytatem**, a gdy odpowiedzi brak, uczciwe „nie ma tego w dokumentach”.